# 4.5 Model Comparison

## Introduction

This notebook compares the four fraud detection models developed in the previous modeling stage:

- Logistic Regression
- Random Forest
- XGBoost
- LightGBM

The objective of this stage is to evaluate the models comparatively rather than independently.

Model selection will not be based on accuracy alone. Since fraud detection involves a strong trade-off between detecting fraudulent transactions and minimizing unnecessary interventions on legitimate transactions, the comparison will focus on several key metrics:

- Precision
- Recall
- F1-Score
- ROC-AUC
- Average Precision (PR-AUC)
- Probability threshold behavior

The analysis will also consider the business context of the Fraud Detection Decision Engine, where detecting fraud is important, but excessive false positives may increase customer friction and operational workload.

Final transaction actions and business thresholds are not defined in this notebook. Those decisions will be handled separately in the Decision Engine stage.

## Comparison Objective

The main objective of this notebook is to answer the following question:

> Which model provides the most suitable fraud probability output for the Fraud Detection Decision Engine?

The comparison will consider both predictive performance and the behavior of each model across different probability thresholds.

The selected model will later be used as the risk scoring component of the Decision Engine.

In [37]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Project paths (absolute to avoid working directory issues)
PROJECT_DIR = Path('/home/rizalkurnia/Projects/DataAnalysis/fraud_detection')
DATA_RAW = PROJECT_DIR / 'data' / 'raw'
OUTPUT_FIG = PROJECT_DIR / 'outputs' / 'figures'
DATA_PROCESSED = PROJECT_DIR / 'data' / 'processed'
MODELS = PROJECT_DIR / 'models'

## 4.5.1 Evaluation Data Preparation

The model comparison stage requires the original test labels and the probability outputs generated by each trained model.

Since the test labels were not exported as a separate artifact during model development, they will be reconstructed using the same train-test split configuration used previously:

- Test size: 20%
- Random state: 42
- Stratified split based on the fraud target

Using the same target data and split configuration ensures that the reconstructed test labels correspond to the probability outputs produced during model evaluation.

In [38]:
from sklearn.model_selection import train_test_split

# Load target column only
y = pd.read_parquet(
    DATA_PROCESSED / "data_optimized.parquet",
    columns=["isFraud"]
)["isFraud"]

# Reconstruct the same test split used during modeling
_, y_test = train_test_split(
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Test samples : {len(y_test):,}")
print(f"Fraud cases  : {y_test.sum():,}")
print(f"Fraud rate   : {y_test.mean():.2%}")

Test samples : 118,108
Fraud cases  : 4,133
Fraud rate   : 3.50%


## 4.5.2 Model Probability Artifacts

The fraud probability outputs generated during model development are loaded for comparative evaluation.

Using the saved probability outputs allows all models to be evaluated against the same reconstructed test set without retraining the models.

In [39]:
import joblib

y_proba_lr = joblib.load(
    MODELS / "logistic_regression_probability.pkl"
)

y_proba_rf = joblib.load(
    MODELS / "random_forest_probability.pkl"
)

y_proba_xgb = joblib.load(
    MODELS / "xgboost_probability.pkl"
)

y_proba_lgbm = joblib.load(
    MODELS / "lightgbm_probability.pkl"
)

print("Probability artifacts loaded successfully.")

print(f"Logistic Regression : {len(y_proba_lr):,}")
print(f"Random Forest       : {len(y_proba_rf):,}")
print(f"XGBoost             : {len(y_proba_xgb):,}")
print(f"LightGBM            : {len(y_proba_lgbm):,}")

Probability artifacts loaded successfully.
Logistic Regression : 118,108
Random Forest       : 118,108
XGBoost             : 118,108
LightGBM            : 118,108


In [40]:
probability_artifacts = {
    "Logistic Regression": y_proba_lr,
    "Random Forest": y_proba_rf,
    "XGBoost": y_proba_xgb,
    "LightGBM": y_proba_lgbm
}

for model_name, probabilities in probability_artifacts.items():
    print(
        f"{model_name:<20} | "
        f"Min: {probabilities.min():.4f} | "
        f"Max: {probabilities.max():.4f} | "
        f"NaN: {pd.isna(probabilities).sum():,}"
    )

Logistic Regression  | Min: 0.0000 | Max: 1.0000 | NaN: 0
Random Forest        | Min: 0.0000 | Max: 1.0000 | NaN: 0
XGBoost              | Min: 0.0003 | Max: 0.9983 | NaN: 0
LightGBM             | Min: 0.0005 | Max: 0.9995 | NaN: 0


## 4.5.3 Overall Model Performance Comparison

The four models are evaluated on the same test set using their saved fraud probability outputs.

To ensure a consistent comparison, classification metrics are recalculated using the default probability threshold of 0.50.

The comparison focuses on:

- Precision
- Recall
- F1-Score
- ROC-AUC
- Average Precision (PR-AUC)

These metrics provide complementary perspectives on model performance, particularly under the highly imbalanced fraud detection setting.

In [42]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

comparison_results = []

for model_name, probabilities in probability_artifacts.items():
    y_pred = (probabilities >= 0.50).astype(int)

    comparison_results.append({
        "Model": model_name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average Precision": average_precision_score(y_test, probabilities)
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,Model,Precision,Recall,F1 Score,ROC-AUC,Average Precision
0,Logistic Regression,0.131543,0.724171,0.222644,0.859537,0.417799
1,Random Forest,0.944009,0.407936,0.569691,0.938780,0.736484
2,XGBoost,0.913343,0.448827,0.601882,0.933282,0.690947
3,LightGBM,0.234515,0.818050,0.364528,0.935411,0.646803


## 4.5.4 Ensemble Comparison

In addition to comparing individual models, this stage evaluates whether combining multiple model probability outputs can produce a better trade-off between Recall and Precision.

The main objective of the ensemble is:

> Improve fraud Recall relative to Random Forest and XGBoost without causing an excessive decline in Precision.

A simple probability averaging approach is used first as a transparent baseline ensemble.

Two ensemble candidates are evaluated:

- Random Forest + XGBoost
- Random Forest + XGBoost + LightGBM

Weighted ensembles are not introduced at this stage, so any improvement can be attributed to the combination of model outputs rather than manually selected weights.

In [43]:
ensemble_probabilities = {
    "RF + XGBoost": (
        y_proba_rf +
        y_proba_xgb
    ) / 2,

    "RF + XGBoost + LightGBM": (
        y_proba_rf +
        y_proba_xgb +
        y_proba_lgbm
    ) / 3
}

In [44]:
ensemble_results = []

for model_name, probabilities in ensemble_probabilities.items():
    y_pred = (probabilities >= 0.50).astype(int)

    ensemble_results.append({
        "Model": model_name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average Precision": average_precision_score(y_test, probabilities)
    })

ensemble_df = pd.DataFrame(ensemble_results)

ensemble_df

,Model,Precision,Recall,F1 Score,ROC-AUC,Average Precision
0,RF + XGBoost,0.939489,0.435761,0.595372,0.945952,0.739149
1,RF + XGBoost + LightGBM,0.862123,0.532543,0.658391,0.944228,0.715845


## 4.5.5 Ensemble Threshold Analysis

The three-model ensemble showed the strongest overall trade-off at the default threshold of 0.50.

However, the final model comparison should not rely only on a single classification threshold.

This section evaluates how Precision, Recall, and F1-Score change across different probability thresholds for the ensemble of:

- Random Forest
- XGBoost
- LightGBM

The objective is to understand whether the ensemble can achieve higher fraud Recall while maintaining an acceptable level of Precision.

In [45]:
import numpy as np

ensemble_proba = ensemble_probabilities["RF + XGBoost + LightGBM"]

thresholds = np.arange(0.10, 0.91, 0.05)

ensemble_threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (ensemble_proba >= threshold).astype(int)

    ensemble_threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, y_pred_threshold),
        "Recall": recall_score(y_test, y_pred_threshold),
        "F1 Score": f1_score(y_test, y_pred_threshold)
    })

ensemble_threshold_df = pd.DataFrame(ensemble_threshold_results)

ensemble_threshold_df

,Threshold,Precision,Recall,F1 Score
0,0.10,0.114667,0.926688,0.204082
1,0.15,0.173368,0.880474,0.289695
2,0.20,0.249046,0.836680,0.383838
3,0.25,0.348139,0.785144,0.482384
4,0.30,0.479117,0.735543,0.580263
5,0.35,0.617031,0.676748,0.645511
6,0.40,0.736857,0.627389,0.677731
7,0.45,0.812798,0.577789,0.675435
8,0.50,0.862123,0.532543,0.658391
9,0.55,0.896612,0.493104,0.636278


## 4.5.6 Final Candidate Comparison

The final comparison includes both individual models and ensemble candidates.

For individual models, the default classification threshold of 0.50 is retained to preserve consistency with the original model evaluation.

For the three-model ensemble, a representative threshold of 0.40 is used because it provides a stronger Recall–Precision trade-off for the business objective.

This threshold is used only for model comparison and does not represent the final business decision threshold.

In [46]:
final_comparison_results = []

# Individual models at threshold 0.50
for model_name, probabilities in probability_artifacts.items():
    y_pred = (probabilities >= 0.50).astype(int)

    final_comparison_results.append({
        "Model": model_name,
        "Threshold": 0.50,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average Precision": average_precision_score(y_test, probabilities)
    })

# RF + XGBoost at threshold 0.50
rf_xgb_proba = ensemble_probabilities["RF + XGBoost"]
rf_xgb_pred = (rf_xgb_proba >= 0.50).astype(int)

final_comparison_results.append({
    "Model": "RF + XGBoost",
    "Threshold": 0.50,
    "Precision": precision_score(y_test, rf_xgb_pred),
    "Recall": recall_score(y_test, rf_xgb_pred),
    "F1 Score": f1_score(y_test, rf_xgb_pred),
    "ROC-AUC": roc_auc_score(y_test, rf_xgb_proba),
    "Average Precision": average_precision_score(y_test, rf_xgb_proba)
})

# RF + XGBoost + LightGBM at threshold 0.40
ensemble_pred_040 = (ensemble_proba >= 0.40).astype(int)

final_comparison_results.append({
    "Model": "RF + XGBoost + LightGBM",
    "Threshold": 0.40,
    "Precision": precision_score(y_test, ensemble_pred_040),
    "Recall": recall_score(y_test, ensemble_pred_040),
    "F1 Score": f1_score(y_test, ensemble_pred_040),
    "ROC-AUC": roc_auc_score(y_test, ensemble_proba),
    "Average Precision": average_precision_score(y_test, ensemble_proba)
})

final_comparison_df = pd.DataFrame(final_comparison_results)

final_comparison_df

,Model,Threshold,Precision,Recall,F1 Score,ROC-AUC,Average Precision
0,Logistic Regression,0.5,0.131543,0.724171,0.222644,0.859537,0.417799
1,Random Forest,0.5,0.944009,0.407936,0.569691,0.938780,0.736484
2,XGBoost,0.5,0.913343,0.448827,0.601882,0.933282,0.690947
3,LightGBM,0.5,0.234515,0.818050,0.364528,0.935411,0.646803
4,RF + XGBoost,0.5,0.939489,0.435761,0.595372,0.945952,0.739149
5,RF + XGBoost + LightGBM,0.4,0.736857,0.627389,0.677731,0.944228,0.715845


## 4.5.7 Best-F1 Threshold Comparison

To ensure a fair technical comparison, each candidate model is evaluated at the threshold that produces its highest F1-Score within the previously tested threshold range.

This comparison represents a model-centric operating point rather than a final business threshold.

The purpose is to compare each candidate under its strongest Precision–Recall balance before applying business-oriented priorities in the next analysis.

In [47]:
import joblib

threshold_lr = pd.read_csv(
    MODELS / "logistic_regression_threshold.csv"
)

threshold_rf = joblib.load(
    MODELS / "random_forest_threshold.pkl"
)

threshold_xgb = joblib.load(
    MODELS / "xgboost_threshold_result.pkl"
)

threshold_lgbm = joblib.load(
    MODELS / "lightgbm_threshold_result.pkl"
)

print("Threshold artifacts loaded successfully.")

Threshold artifacts loaded successfully.


In [48]:
for name, df in {
    "Logistic Regression": threshold_lr,
    "Random Forest": threshold_rf,
    "XGBoost": threshold_xgb,
    "LightGBM": threshold_lgbm
}.items():
    print(f"\n{name}")
    print(df.head())
    print("Columns:", list(df.columns))


Logistic Regression
   Threshold  Precision    Recall  F1 Score
0       0.10   0.040952  0.984999  0.078634
1       0.15   0.046797  0.967094  0.089274
2       0.20   0.052694  0.951125  0.099855
3       0.25   0.059611  0.924510  0.112000
4       0.30   0.069517  0.899589  0.129061
Columns: ['Threshold', 'Precision', 'Recall', 'F1 Score']

Random Forest
   Threshold  Precision    Recall  F1 Score
0       0.10   0.469278  0.779821  0.585947
1       0.15   0.654260  0.706025  0.679157
2       0.20   0.754574  0.648681  0.697632
3       0.25   0.820871  0.597629  0.691683
4       0.30   0.864855  0.554319  0.675612
Columns: ['Threshold', 'Precision', 'Recall', 'F1 Score']

XGBoost
   Threshold  Precision    Recall  F1 Score
0       0.10   0.475476  0.713041  0.570516
1       0.15   0.616792  0.652311  0.634055
2       0.20   0.713103  0.604404  0.654269
3       0.25   0.776045  0.565933  0.654540
4       0.30   0.826945  0.537624  0.651613
Columns: ['Threshold', 'Precision', 'Recall', '

In [49]:
thresholds = np.arange(0.10, 0.91, 0.05)

candidate_probabilities = {
    "Logistic Regression": y_proba_lr,
    "Random Forest": y_proba_rf,
    "XGBoost": y_proba_xgb,
    "LightGBM": y_proba_lgbm,
    "RF + XGBoost": ensemble_probabilities["RF + XGBoost"],
    "RF + XGBoost + LightGBM": ensemble_proba
}

best_f1_results = []

for model_name, probabilities in candidate_probabilities.items():

    model_threshold_results = []

    for threshold in thresholds:
        y_pred = (probabilities >= threshold).astype(int)

        model_threshold_results.append({
            "Threshold": threshold,
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred)
        })

    model_threshold_df = pd.DataFrame(model_threshold_results)

    best_row = model_threshold_df.loc[
        model_threshold_df["F1 Score"].idxmax()
    ]

    best_f1_results.append({
        "Model": model_name,
        "Best Threshold": best_row["Threshold"],
        "Precision": best_row["Precision"],
        "Recall": best_row["Recall"],
        "F1 Score": best_row["F1 Score"],
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "Average Precision": average_precision_score(
            y_test,
            probabilities
        )
    })

best_f1_comparison_df = pd.DataFrame(best_f1_results)

best_f1_comparison_df

,Model,Best Threshold,Precision,Recall,F1 Score,ROC-AUC,Average Precision
0,Logistic Regression,0.85,0.472674,0.393419,0.429420,0.859537,0.417799
1,Random Forest,0.20,0.754574,0.648681,0.697632,0.938780,0.736484
2,XGBoost,0.25,0.776045,0.565933,0.654540,0.933282,0.690947
3,LightGBM,0.85,0.711210,0.543431,0.616102,0.935411,0.646803
4,RF + XGBoost,0.20,0.760011,0.642874,0.696553,0.945952,0.739149
5,RF + XGBoost + LightGBM,0.40,0.736857,0.627389,0.677731,0.944228,0.715845


## 4.5.8 Business-Oriented Candidate Comparison

After the technical comparison, two candidates remain particularly relevant for the Fraud Detection Decision Engine:

- Random Forest
- Random Forest + XGBoost + LightGBM Ensemble

The comparison now shifts from aggregate model metrics toward operational outcomes.

For each candidate, the analysis evaluates:

- True Positives: fraudulent transactions correctly detected
- False Negatives: fraudulent transactions that remain undetected
- False Positives: legitimate transactions incorrectly flagged
- True Negatives: legitimate transactions correctly classified
- Total flagged transactions

Random Forest is evaluated at a threshold of 0.20, while the three-model ensemble is evaluated at 0.40.

These thresholds represent comparative operating points identified during model analysis and are not final Decision Engine thresholds.

In [50]:
from sklearn.metrics import confusion_matrix

business_candidates = {
    "Random Forest": {
        "probabilities": y_proba_rf,
        "threshold": 0.20
    },
    "RF + XGBoost + LightGBM": {
        "probabilities": ensemble_proba,
        "threshold": 0.40
    }
}

business_results = []

for model_name, config in business_candidates.items():

    threshold = config["threshold"]
    probabilities = config["probabilities"]

    y_pred = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    business_results.append({
        "Model": model_name,
        "Threshold": threshold,
        "True Positives": tp,
        "False Negatives": fn,
        "False Positives": fp,
        "True Negatives": tn,
        "Flagged Transactions": tp + fp
    })

business_comparison_df = pd.DataFrame(business_results)

business_comparison_df

,Model,Threshold,True Positives,False Negatives,False Positives,True Negatives,Flagged Transactions
0,Random Forest,0.2,2703,1430,915,113060,3618
1,RF + XGBoost + LightGBM,0.4,2593,1540,926,113049,3519


## 4.5.9 Final Model Selection

Based on the technical and business-oriented comparison, Random Forest is selected as the primary model for the Fraud Detection Decision Engine.

The model demonstrated the strongest overall trade-off between fraud detection capability and operational efficiency.

At a threshold of 0.20, Random Forest achieved:

- Precision: 75.46%
- Recall: 64.87%
- F1-Score: 69.76%
- ROC-AUC: 0.9388
- Average Precision: 0.7365

Compared with the three-model ensemble at its representative operating point, Random Forest detected 110 additional fraudulent transactions while producing 11 fewer false positives.

This result is particularly important because the project prioritizes reducing False Negatives while maintaining an acceptable level of False Positives.

Although ensemble approaches achieved slightly higher ROC-AUC and Average Precision in some configurations, the improvements were not large enough to justify the additional model complexity.

Therefore, Random Forest provides the most suitable balance between:

- Fraud Recall
- Precision
- Overall Precision–Recall trade-off
- Operational efficiency
- Model simplicity

The threshold of 0.20 used in this comparison should not be interpreted as the final business decision threshold.

The next stage will use the Random Forest fraud probability output to design a multi-level Decision Engine, where probability scores are translated into operational actions such as approval, additional authentication, or manual review.